# NLP Assignment 2025/26 - PoliMillionaire Speech QuizBot

**Project.** Local speech-mode chatbot experiment for the online quiz *Who wants to be a PoliMillionaire?*.

**Group members.**

- Salvatore Mariano Librici - salvatoremariano.librici@mail.polimi.it
- Rong Huang - rong.huang@mail.polimi.it
- Soumyadeep Sharma - soumyadeep.sharma@mail.polimi.it
- Merve Rana Kizil - merverana.kizil@mail.polimi.it
- Selahattin Cem Ozturk - selahattincem.ozturk@mail.polimi.it

**Videos.**

- [5-minute notebook presentation](https://www.youtube.com/watch?v=evtf6K6-Tvg)
- [Optional long system demo](https://www.youtube.com/watch?v=pEzU3mz5Gos)

**Coding assistants statement.** OpenAI ChatGPT/Codex was used to help scaffold, debug, document, and refine parts of this notebook. The final design choices, experiments, execution, and interpretation remain the responsibility of the group.

This notebook is a separate speech-mode experiment. It keeps the text benchmark notebook clean and focuses on the audio interface:

1. fetch question and answer-option audio from the official game client;
2. transcribe the audio locally with Whisper;
3. answer with a local open-weight text model;
4. submit the final `option_id` through the official API client.

No external LLM API is used. Whisper and the answer model run locally in Colab.


## 1. Global Settings

All settings for the speech experiment are centralized here.


In [34]:
# Centralized, all settings are. Scattered through the notebook, they are not.
API_URL = 'http://131.175.15.22:51111/'
USERNAME = 'MarianoAkaMery'
PASSWORD = 'Test1234!'

DRIVE_PROJECT_DIR = '/content/gdrive/MyDrive/[2025-26] - 088946 - NLP PROJECT'

COMPETITION_KEY = 'science_nature'
COMPETITIONS = {
    'entertainment': {'id': 0, 'name': 'Entertainment'},
    'ancient_history_politics': {'id': 1, 'name': 'Ancient History and Politics'},
    'science_nature': {'id': 2, 'name': 'Science and Nature'},
    'maths': {'id': 3, 'name': 'Maths'},
    'philosophy_psychology': {'id': 4, 'name': 'Philosophy and Psychology'},
    'news': {'id': 5, 'name': 'News'},
}
COMPETITION_ID = COMPETITIONS[COMPETITION_KEY]['id']
GAME_MODE = 'speech'

ANSWER_MODEL_NAME = 'Qwen/Qwen2.5-3B-Instruct'
WHISPER_MODEL_NAME = 'openai/whisper-small'
LOAD_IN_4BIT = True
MAX_NEW_TOKENS = 2

MAX_LEVELS_TO_PLAY = 15
DELAY_BETWEEN_QUESTIONS_S = 1.0
SAVE_RUN_LOG = True
PRINT_TRANSCRIPTS = True
PRINT_WRONG_QUESTION_DEBUG = True

INSTALL_DEPENDENCIES = True
RUN_API_CHECK = True
RUN_MODEL_WARMUP = True
RUN_FULL_SPEECH_GAME = False
RUN_ALL_CATEGORIES_SPEECH_BENCHMARK = True
SPEECH_BENCHMARK_CATEGORY_KEYS = [
    'entertainment',
    'ancient_history_politics',
    'science_nature',
    'maths',
    'philosophy_psychology',
    'news',
]
SPEECH_BENCHMARK_RUNS_PER_CATEGORY = 3

print('Speech settings loaded.')


Speech settings loaded.


## 2. Colab Setup

This mounts Google Drive and makes `millionaire_client` importable.


In [35]:
# Mounted, Google Drive is. Found, the project folder must be.
import os
import sys
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/gdrive/')
    IN_COLAB = True
except Exception:
    IN_COLAB = False

PROJECT_DIR = DRIVE_PROJECT_DIR if IN_COLAB else os.getcwd()
if PROJECT_DIR not in sys.path:
    sys.path.append(PROJECT_DIR)

print('IN_COLAB:', IN_COLAB)
print('PROJECT_DIR:', PROJECT_DIR)
print('millionaire_client exists:', os.path.isdir(os.path.join(PROJECT_DIR, 'millionaire_client')))


Drive already mounted at /content/gdrive/; to attempt to forcibly remount, call drive.mount("/content/gdrive/", force_remount=True).
IN_COLAB: True
PROJECT_DIR: /content/gdrive/MyDrive/[2025-26] - 088946 - NLP PROJECT
millionaire_client exists: True


## 3. Dependencies

The speech notebook needs local ASR and local text generation packages.


In [36]:
# Installed, speech and model dependencies are. Locally, everything runs.
if INSTALL_DEPENDENCIES:
    import subprocess
    subprocess.check_call([
        sys.executable,
        '-m',
        'pip',
        'install',
        '-q',
        'transformers',
        'accelerate',
        'sentencepiece',
        'bitsandbytes',
        'torch',
        'soundfile',
        'datasets',
        'torchaudio',
    ])
    print('Dependencies installed or already available.')
else:
    print('Dependency installation skipped by settings.')

import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: no GPU detected. CPU inference will likely timeout.')

Dependencies installed or already available.
CUDA available: True
GPU: Tesla T4


## 4. Official Game Client

Only the course-provided client is used for game interaction.


In [37]:
# Imported, the official client is. Invented endpoints, used they are not.
from millionaire_client import MillionaireClient, AuthenticationError, MillionaireError, GameError

client = MillionaireClient(API_URL)
user = client.login(USERNAME, PASSWORD)
print(f'Logged in as {user.username} (role={user.role})')

def competition_key_from_name(name):
    clean = str(name).strip().lower().replace('&', 'and')
    clean = clean.replace('/', ' ').replace('-', ' ')
    clean = '_'.join(clean.split())
    aliases = {
        'ancient_history_and_politics': 'ancient_history_politics',
        'philosophy_and_psychology': 'philosophy_psychology',
    }
    return aliases.get(clean, clean)

competitions = client.competitions.list_all()
for comp in competitions:
    key = competition_key_from_name(comp.name)
    COMPETITIONS[key] = {'id': comp.id, 'name': comp.name}

COMPETITION_ID = COMPETITIONS[COMPETITION_KEY]['id']
print(f'Active competition: {COMPETITIONS[COMPETITION_KEY]["name"]} (id={COMPETITION_ID})')

if RUN_API_CHECK:
    print('Available competitions:')
    for comp in competitions:
        print(f'  {comp.id}: {comp.name} | max_levels={comp.max_levels}')
else:
    print('API check skipped by settings.')


Logged in as MarianoAkaMery (role=student)
Available competitions:
  0: Entertainment | max_levels=15
  1: Ancient History and Politics | max_levels=15
  2: Science and Nature | max_levels=15
  3: Maths | max_levels=15


## 5. Category Prompt and Answer Parsing

Whisper transcripts are noisy, so the prompt asks for only one answer letter. The letter is then mapped to the option id delivered by the game state.


In [38]:
# Built, the prompt is. Parsed, only valid answer letters are.
import re

CATEGORY_PROMPTS = {
    'general': 'You are a careful multiple-choice quiz expert. Use the transcript and answer choices to select the single best answer.',
    'entertainment': 'You are a careful quiz expert on music, movies, television, celebrities, awards, and pop culture.',
    'ancient_history_politics': 'You are a careful quiz expert on ancient history and politics, especially Rome, Greece, empires, rulers, and institutions.',
    'science_nature': 'You are a careful quiz expert on science and nature, including chemistry, biology, physics, earth science, astronomy, and units.',
    'maths': 'You are a careful mathematics and statistics tutor. Compute exactly and choose the answer choice that matches.',
    'philosophy_psychology': 'You are a careful quiz expert on philosophy and psychology, including philosophers, schools of thought, ethics, logic, cognitive psychology, clinical concepts, named experiments, disorders, therapies, and theories. Pay attention to definitions and similar-sounding theories.',
    'news': 'You are answering a news-based multiple-choice quiz. Many questions refer to specific recent articles, dates, parties, offices, institutions, unions, government departments, and named public figures. If the event is too recent or unfamiliar, compare all four options and reason from country, location, organization, role, acronym, date, and cause-effect wording. Do not rely on one keyword only.',
}

def build_speech_prompt(question_text, option_texts, competition_key=COMPETITION_KEY):
    option_lines = '\n'.join(f'{chr(65 + idx)}) {text}' for idx, text in enumerate(option_texts))
    category_prompt = CATEGORY_PROMPTS.get(competition_key, CATEGORY_PROMPTS['general'])
    return f"""{category_prompt}

You are playing a multiple-choice quiz from speech transcripts.
The transcript may contain small ASR mistakes.
Choose the single best option.
Return only one letter: A, B, C, or D.
Do not explain.

Question transcript:
{question_text}

Option transcripts:
{option_lines}

Answer:"""

def parse_answer_letter(raw_text, num_options=4):
    valid_letters = [chr(65 + idx) for idx in range(num_options)]
    letter_group = ''.join(valid_letters)
    cleaned = str(raw_text).strip().upper()
    if not cleaned:
        return None
    direct = re.match(rf'^\s*([{letter_group}])(?:\s|\)|\.|:|$)', cleaned)
    if direct:
        return direct.group(1)
    patterns = [
        rf'FINAL\s+ANSWER\s*(?:IS|:)?\s*([{letter_group}])\b',
        rf'ANSWER\s*(?:IS|:)?\s*([{letter_group}])\b',
        rf'OPTION\s*([{letter_group}])\b',
    ]
    for pattern in patterns:
        matches = re.findall(pattern, cleaned)
        if matches:
            return matches[-1]
    standalone = re.findall(rf'(?<![A-Z])([{letter_group}])(?![A-Z])', cleaned)
    return standalone[-1] if standalone else None

def letter_to_option_id(letter, question):
    if letter is None:
        return None
    index = ord(letter) - ord('A')
    if 0 <= index < len(question.options):
        return question.options[index].id
    return None


## 6. Local Whisper ASR

The server provides WAV audio bytes. This cell saves each clip temporarily and transcribes it with a local Whisper pipeline.


In [39]:
# Transcribed, audio clips are. Locally, Whisper runs.
import tempfile
from transformers import WhisperProcessor, WhisperForConditionalGeneration
import soundfile as sf
import io # Added for in-memory audio reading
import torch # For tensor operations
import torchaudio # For resampling

# Global variables for processor and model
whisper_processor = None
whisper_model = None
target_sampling_rate = None

def load_asr():
    global whisper_processor, whisper_model, target_sampling_rate
    if whisper_processor is not None and whisper_model is not None:
        return whisper_processor, whisper_model

    device = "cuda:0" if torch.cuda.is_available() else "cpu"

    whisper_processor = WhisperProcessor.from_pretrained(WHISPER_MODEL_NAME)
    whisper_model = WhisperForConditionalGeneration.from_pretrained(WHISPER_MODEL_NAME).to(device)
    # Required for Whisper to work without forced language
    whisper_model.config.forced_decoder_ids = None

    target_sampling_rate = whisper_processor.feature_extractor.sampling_rate

    return whisper_processor, whisper_model

def transcribe_wav_bytes(audio_bytes, label):
    processor, model = load_asr() # Get processor and model
    try:
        audio_buffer = io.BytesIO(audio_bytes)
        audio_array, original_sampling_rate = sf.read(audio_buffer, dtype='float32')

        # If the audio is stereo, convert to mono by taking the mean of channels
        if audio_array.ndim > 1:
            audio_array = audio_array.mean(axis=1)

        # Resample the audio to the target sampling rate if necessary
        if original_sampling_rate != target_sampling_rate:
            print(f"Resampling audio from {original_sampling_rate} Hz to {target_sampling_rate} Hz for {label}")
            # Convert to torch tensor for torchaudio
            audio_tensor = torch.from_numpy(audio_array)
            resampler = torchaudio.transforms.Resample(
                orig_freq=original_sampling_rate,
                new_freq=target_sampling_rate
            )
            audio_array = resampler(audio_tensor).numpy()
            # The sampling_rate used by the feature extractor should be the target_sampling_rate after resampling
            sampling_rate_for_processor = target_sampling_rate
        else:
            sampling_rate_for_processor = original_sampling_rate

        # Manually process audio through the feature extractor
        # Explicitly set return_attention_mask=True, padding="max_length" and max_length to 30 seconds worth of samples
        # This is crucial for matching Whisper's fixed input length requirement of 3000 mel frames
        inputs = processor.feature_extractor(
            audio_array,
            sampling_rate=sampling_rate_for_processor,
            return_tensors="pt",
            padding="max_length", # Pad to max_length
            max_length=int(sampling_rate_for_processor * 30), # 30 seconds worth of samples (target for 3000 mel frames)
            return_attention_mask=True # Often required for num_frames
        )

        # Move inputs to the correct device
        input_features = inputs.input_features.to(model.device)
        attention_mask = inputs.attention_mask.to(model.device) if inputs.attention_mask is not None else None

        # Generate token IDs
        predicted_ids = model.generate(input_features=input_features, attention_mask=attention_mask)

        # Decode token IDs to text
        transcription = processor.tokenizer.decode(predicted_ids[0], skip_special_tokens=True)
        return transcription.strip()
    except Exception as e:
        print(f"Error during audio transcription: {e}")
        raise # Re-raise the error after printing for better visibility

## 7. Local Answer Model

This is the same type of local answer model used by the text notebook.


In [40]:
# Loaded locally, the answer model is. External LLM API, used it is not.
import time

class LocalCausalLMAnswerer:
    def __init__(self, model_name, max_new_tokens=8):
        self.model_name = model_name
        self.max_new_tokens = max_new_tokens
        self.tokenizer = None
        self.model = None

    def load(self):
        from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
        if self.model is not None:
            return

        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
        quantization_config = None
        if LOAD_IN_4BIT:
            quantization_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_use_double_quant=True,
                bnb_4bit_quant_type='nf4',
                bnb_4bit_compute_dtype=torch.float16,
            )

        self.model = AutoModelForCausalLM.from_pretrained(
            self.model_name,
            device_map={'': 0} if torch.cuda.is_available() else 'cpu',
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
            quantization_config=quantization_config,
        )
        self.model.eval()

    def generate_text(self, prompt, max_new_tokens=None):
        self.load()
        inputs = self.tokenizer(prompt, return_tensors='pt').to(self.model.device)

        start = time.time()
        output_ids = self.model.generate(
            **inputs,
            max_new_tokens=max_new_tokens or self.max_new_tokens,
            do_sample=False,
            pad_token_id=self.tokenizer.eos_token_id,
        )
        elapsed_s = time.time() - start

        generated_ids = output_ids[0][inputs['input_ids'].shape[-1]:]
        raw = self.tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
        return raw, elapsed_s

    def answer(self, question_text, option_texts, question_state, competition_key=COMPETITION_KEY):
        prompt = build_speech_prompt(question_text, option_texts, competition_key)
        raw, elapsed_s = self.generate_text(prompt)
        letter = parse_answer_letter(raw, num_options=len(option_texts))
        option_id = letter_to_option_id(letter, question_state)

        return {
            'model': self.model_name,
            'raw': raw,
            'letter': letter,
            'option_id': option_id,
            'elapsed_s': elapsed_s,
        }

answer_model = LocalCausalLMAnswerer(ANSWER_MODEL_NAME, max_new_tokens=MAX_NEW_TOKENS)
print('Answer model:', ANSWER_MODEL_NAME)


Answer model: Qwen/Qwen2.5-3B-Instruct


## 8. Warm-up

Models are loaded before a real speech game starts. In speech mode the timer starts after option D is requested, so model loading during the game would be fatal.


In [41]:
# Warmed up, ASR and answer model are. During the game, loaded they should not be.
import gc

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

if RUN_MODEL_WARMUP:
    print('Loading Whisper ASR...')
    # Modified to call load_asr without assigning to asr_pipeline
    load_asr()
    print('Whisper ready.')

    print('Loading answer model...')
    answer_model.load()
    print('Answer model ready.')

    print('Warm-up complete.')
else:
    print('Warm-up skipped by settings.')

Loading Whisper ASR...


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Whisper ready.
Loading answer model...


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Answer model ready.
Warm-up complete.


In [ ]:
# Cleaned deterministically, speech transcripts are. Invented content, avoided it is.
def clean_question_transcript(question_text):
    text = str(question_text).strip()
    text = re.sub(r'\s+', ' ', text)
    return text

def clean_option_transcript(option_text, expected_letter):
    text = str(option_text).strip()
    text = re.sub(r'\s+', ' ', text)

    # Common ASR variants for option markers, removed before the option text is used.
    marker = r'(?:option|opsin|opsen|obsin|absin|upson|upsin|ops\s+in|ups\s+in|ops\s+and|ups\s+and)'
    text = re.sub(r'^' + marker + r'\s*' + expected_letter + r'\s*[\.,:;\-\)]?\s*', '', text, flags=re.I)
    text = re.sub(r'^' + marker + r'\s*[\.,:;\-\)]?\s*', '', text, flags=re.I)

    # Remove a leading expected letter marker.
    text = re.sub(r'^' + expected_letter + r'\s*[\.,:;\-\)]\s*', '', text, flags=re.I)

    # Remove one spurious leading answer-letter marker, often spoken by the server or hallucinated by ASR.
    text = re.sub(r'^[ABCD]\s*[\.,:;\-\)]\s+', '', text, flags=re.I)

    # Clean common filler/noise at the beginning.
    text = re.sub(r'^(?:u+h+|um+|uh+|er+)\W*\s*', '', text, flags=re.I)
    text = re.sub(r'^(ha\s*){2,}', '', text, flags=re.I)
    text = text.strip(' ,.;:-?!')

    # If ASR only captured an option marker or a stray letter, keep a neutral placeholder.
    if not text or re.fullmatch(r'(?:option\s*)?[A-E]', text, flags=re.I):
        return '[unreadable option]'
    return text

def deterministic_cleanup_speech_item(question_text, option_texts):
    cleaned_question = clean_question_transcript(question_text)
    cleaned_options = [
        clean_option_transcript(option_text, chr(65 + idx))
        for idx, option_text in enumerate(option_texts)
    ]
    return cleaned_question, cleaned_options, {
        'method': 'deterministic_cleanup',
        'question': cleaned_question,
        'options': cleaned_options,
    }


## 9. Full Speech Game

This starts one real speech-mode game. Audio clips are transcribed first, then the answer model chooses the option id.


In [42]:
# Played, a full speech game is. Transcribed, each audio option becomes.
import json
from datetime import UTC, datetime

OPTION_LABELS = ['A', 'B', 'C', 'D']

def fetch_and_transcribe_current_question(game):
    question_audio = game.fetch_audio_question()
    question_text = transcribe_wav_bytes(question_audio, f'level_{game.current_level}_question')

    option_texts = []
    option_audio_lengths = []
    for option_index in range(4):
        option_audio = game.fetch_audio_option_next()
        option_audio_lengths.append(len(option_audio))
        option_text = transcribe_wav_bytes(option_audio, f'level_{game.current_level}_option_{OPTION_LABELS[option_index]}')
        option_texts.append(option_text)

    game.refresh_state()
    return question_text, option_texts, len(question_audio), option_audio_lengths

def play_full_speech_game(competition_key=COMPETITION_KEY):
    run_log = []
    competition_id = COMPETITIONS[competition_key]['id']
    game = client.game.start(competition_id=competition_id, mode='speech')
    print(f'Started speech session: {game.session_id}')

    while game.in_progress and game.current_level <= MAX_LEVELS_TO_PLAY:
        question_state = game.current_question
        if question_state is None:
            break

        question_text, option_texts, question_audio_len, option_audio_lens = fetch_and_transcribe_current_question(game)
        time_before_s = game.time_remaining

        cleaned_question_text, cleaned_option_texts, cleanup = deterministic_cleanup_speech_item(
            question_text,
            option_texts,
        )

        final_question_text = cleaned_question_text
        final_option_texts = cleaned_option_texts

        if PRINT_TRANSCRIPTS:
            print(f'\nLevel {game.current_level} transcripts')
            print('Q raw:', question_text)
            for idx, opt_text in enumerate(option_texts):
                print(f'  {OPTION_LABELS[idx]}) raw: {opt_text}')
            if cleanup is not None:
                print('Q cleaned:', cleaned_question_text)
                for idx, opt_text in enumerate(cleaned_option_texts):
                    print(f'  {OPTION_LABELS[idx]}) cleaned: {opt_text}')
            print(f'Time remaining after audio+ASR: {time_before_s:.1f}s')

        decision = answer_model.answer(final_question_text, final_option_texts, question_state, competition_key)
        if cleanup is not None:
            decision['cleanup'] = cleanup
        chosen_id = decision['option_id'] if decision['option_id'] is not None else question_state.options[0].id
        chosen_letter = decision['letter'] if decision['letter'] is not None else 'A'
        result = game.answer(chosen_id)

        print(
            f"Level {game.current_level} | "
            f"chosen={chosen_letter}:{chosen_id} | "
            f"correct={result.correct} | "
            f"timeout={result.timed_out} | "
            f"model_time={decision['elapsed_s']:.2f}s | "
            f"time_before_answer={time_before_s:.1f}s | "
            f"earned={result.earned_amount}"
        )

        if PRINT_WRONG_QUESTION_DEBUG and result.correct is False:
            print('  Wrong speech-question debug:')
            print(f'    transcript Q: {question_text}')
            for idx, opt_text in enumerate(option_texts):
                print(f'    {OPTION_LABELS[idx]}) id={question_state.options[idx].id} | transcript={opt_text}')
            print(f"    raw={decision['raw']!r}")

        run_log.append({
            'timestamp': datetime.now(UTC).isoformat(),
            'session_id': game.session_id,
            'competition_key': competition_key,
            'competition_name': COMPETITIONS[competition_key]['name'],
            'mode': 'speech',
            'level': game.current_level,
            'question_id': question_state.id,
            'question_transcript': question_text,
            'option_transcripts': option_texts,
            'cleaned_question': cleaned_question_text,
            'cleaned_options': cleaned_option_texts,
            'cleanup': cleanup,
            'question_audio_bytes': question_audio_len,
            'option_audio_bytes': option_audio_lens,
            'time_remaining_before_answer_s': time_before_s,
            'chosen_option_id': chosen_id,
            'chosen_letter': chosen_letter,
            'decision': decision,
            'correct': result.correct,
            'timed_out': result.timed_out,
            'game_over': result.game_over,
            'earned_amount': result.earned_amount,
        })

        if result.game_over or result.timed_out:
            break

        time.sleep(DELAY_BETWEEN_QUESTIONS_S)

    print(f'Final level: {game.current_level}')
    print(f'Final earned amount: {game.earned_amount}')
    return game, run_log

if RUN_FULL_SPEECH_GAME:
    final_game, speech_run_log = play_full_speech_game(COMPETITION_KEY)
else:
    final_game, speech_run_log = None, []
    print('Full speech game skipped by settings.')


Started speech session: 152139
Resampling audio from 24000 Hz to 16000 Hz for level_1_question
Resampling audio from 24000 Hz to 16000 Hz for level_1_option_A
Resampling audio from 24000 Hz to 16000 Hz for level_1_option_B
Resampling audio from 24000 Hz to 16000 Hz for level_1_option_C
Resampling audio from 24000 Hz to 16000 Hz for level_1_option_D

Level 1 transcripts
Q: Which of these organisms is classified as a decomposer?
  A) Option A. Mouse.
  B) Opsin B. Grass!
  C) Option C.
  D) Option D, mushroom. I thought he said he would log, log, log, log.
Time remaining after audio+ASR: 28.2s
Level 2 | chosen=D:3 | correct=True | timeout=False | model_time=1.23s | time_before_answer=28.2s | earned=100
Resampling audio from 24000 Hz to 16000 Hz for level_2_question
Resampling audio from 24000 Hz to 16000 Hz for level_2_option_A
Resampling audio from 24000 Hz to 16000 Hz for level_2_option_B
Resampling audio from 24000 Hz to 16000 Hz for level_2_option_C
Resampling audio from 24000 Hz to 

## 10. All-Category Speech Benchmark

This optional section runs speech-mode games across all categories. Speech mode is heavier than text mode, so keep the number of runs small when testing.


In [ ]:
# Compared, all speech categories are. Heavy, speech mode is.
def summarize_speech_run(game, log):
    correct_answers = sum(1 for item in log if item.get('correct') is True)
    timed_out = any(item.get('timed_out') for item in log)
    last_entry = log[-1] if log else {}
    return {
        'competition_key': last_entry.get('competition_key'),
        'competition_name': last_entry.get('competition_name'),
        'session_id': game.session_id if game else None,
        'questions_answered': len(log),
        'correct_answers': correct_answers,
        'final_level': game.current_level if game else None,
        'earned_amount': game.earned_amount if game else 0,
        'timed_out': timed_out,
    }

speech_benchmark_log = []
speech_benchmark_summary = []

if RUN_ALL_CATEGORIES_SPEECH_BENCHMARK:
    for category_key in SPEECH_BENCHMARK_CATEGORY_KEYS:
        for run_index in range(SPEECH_BENCHMARK_RUNS_PER_CATEGORY):
            print(f"\n=== Speech benchmark {category_key} | run {run_index + 1}/{SPEECH_BENCHMARK_RUNS_PER_CATEGORY} ===")
            game_result, category_log = play_full_speech_game(category_key)
            speech_benchmark_log.extend(category_log)
            speech_benchmark_summary.append(summarize_speech_run(game_result, category_log))
            time.sleep(DELAY_BETWEEN_QUESTIONS_S)

    print('\nSpeech benchmark summary:')
    for row in speech_benchmark_summary:
        print(
            f"{row['competition_name']}: "
            f"correct={row['correct_answers']}/{row['questions_answered']} | "
            f"level={row['final_level']} | "
            f"earned={row['earned_amount']} | "
            f"timeout={row['timed_out']}"
        )
else:
    print('All-category speech benchmark skipped by settings.')


## 11. Save Speech Results

The speech log includes transcripts, timings, model outputs, and final outcomes.


In [43]:
# Saved, the speech run is. Analyzed later, ASR errors can be.
logs_to_save = speech_benchmark_log if 'speech_benchmark_log' in globals() and speech_benchmark_log else speech_run_log

if SAVE_RUN_LOG and logs_to_save:
    log_dir = Path(PROJECT_DIR) / 'runs'
    log_dir.mkdir(parents=True, exist_ok=True)
    output_path = log_dir / f'speech_run_{datetime.now(UTC).strftime("%Y%m%d_%H%M%S")}.json'
    with output_path.open('w', encoding='utf-8') as f:
        json.dump(logs_to_save, f, indent=2, ensure_ascii=False)
    print('Saved speech run log:', output_path)
else:
    print('No speech run log saved.')


Saved speech run log: /content/gdrive/MyDrive/[2025-26] - 088946 - NLP PROJECT/runs/speech_run_20260519_173037.json


## 12. Speech Leaderboard Positions

This checks the current user's speech-mode leaderboard entry for every competition, in the same category order used by the benchmark.


In [44]:
# Checked, our speech leaderboard positions are. Per category, ordered they become.
def find_my_speech_entry(competition_key, limit=100):
    competition_id = COMPETITIONS[competition_key]['id']
    leaderboard = client.leaderboard.get(competition_id=competition_id, limit=limit, mode='speech')
    for rank, entry in enumerate(leaderboard.entries, 1):
        if entry.username == USERNAME:
            return leaderboard.competition.name, rank, entry
    return leaderboard.competition.name, None, None

print(f'Speech leaderboard positions for {USERNAME}:')
for category_key in SPEECH_BENCHMARK_CATEGORY_KEYS:
    competition_name, rank, entry = find_my_speech_entry(category_key)
    if entry is None:
        print(f'{competition_name}: not found in top 100')
    else:
        print(
            f'{competition_name}: rank={rank} | '
            f'score={entry.score} | '
            f'level={entry.reached_level} | '
            f'trials={entry.total_trials}'
        )


Speech leaderboard: Science and Nature
1. ialkbr: 1024000 | level=15
2. gabrielep: 1024000 | level=15
3. shijie: 1024000 | level=15
4. Zero37: 1024000 | level=15
5. IWillBeABillionaire: 1024000 | level=15
6. luca_bordin: 1024000 | level=15
7. supreme_leader: 1024000 | level=15
8. Vercingetorige: 256000 | level=13
9. gioferola: 64000 | level=11
10. ciani: 64000 | level=11
11. gdonninelli: 64000 | level=11
12. gvin: 32000 | level=10
13. lorenzodemontis: 2000 | level=6
14. AndreaL: 1000 | level=5
15. h_raisy: 1000 | level=5
16. MarianoAkaMery: 1000 | level=5 <-- us
17. Andrea Mikhaiel: 500 | level=4
18. TheLastGuessbender: 300 | level=3
19. nicolo: 100 | level=1
20. chia: 100 | level=1


## 13. Final Outcome and Limitations

This notebook evaluates the speech interface as an additional experiment. The full pipeline fetches audio from the official game client, transcribes question and option audio locally with Whisper, applies deterministic transcript cleanup, and then answers with a local open-weight text model.

Speech mode is harder and more variable than text mode because audio fetching, ASR latency, and transcription noise all happen before the answer model is called. The logs therefore include raw transcripts, cleaned transcripts, audio sizes, time remaining after ASR, model outputs, and final game results.

In the final speech benchmark, the best results were observed on Science and Nature and Entertainment. These categories sometimes reached high levels when the transcription was clean enough. Philosophy and Psychology produced medium-to-good results, but remained less stable than in text mode.

Ancient History and Politics, Maths, and News were weaker in speech mode. Ancient History often contains names, places, dynasties, and ancient terms that are easy for ASR to distort. Maths is difficult because formulas, symbols, numbers, and short answer options are not reliably transcribed. News has the same limitation observed in text mode, because many questions refer to very recent article-specific facts, and speech adds an additional layer of transcription noise.

A generative transcript normalizer was tested but removed from the default pipeline because it sometimes changed or truncated answer options. The final version uses deterministic cleanup rules only, so the system removes common ASR artifacts without inventing new content.

Overall, speech mode is useful as an additional architecture and for analyzing the impact of ASR errors, but the text-mode notebook remains the main competitive system because it receives clean question and option text directly from the API.
